In [1]:

from sqlalchemy import text          
from app.db import session as db_session
from app.db.session import get_settings, get_engine

get_settings.cache_clear()
db_session._ENGINES.clear()

settings = get_settings()
engine = get_engine()
print("settings.database_url :", settings.database_url)
print("drivername :", engine.url.drivername)
print("host       :", engine.url.host)
print("port       :", engine.url.port)
print("database   :", engine.url.database)


print("SQLite 전용 분기 :",
      "탔다" if settings.database_url.startswith("sqlite") else "타지 않았다")

print()
try:
    
    with engine.connect() as conn:
        version = conn.execute(text("SELECT version()")).scalar()
    print("SELECT version() →", version[:40], "...")
except Exception as e:
    print(f"접속 실패 — {type(e).__name__}")
    print("  ① 컨테이너가 떠 있는가        : docker compose ps  →  (healthy) 인지 본다")
    print("  ② 포트가 맞는가              : .env 의 5432 와 compose 의 ports 를 대조한다")
    print("  ③ 드라이버가 깔려 있는가      : pip show psycopg")


ModuleNotFoundError: No module named 'app'

In [ ]:
from app.db.init_db import init_db
from app.db.seed import seed_all, count_rows
from app.db.session import session_scope

init_db()      

with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'public' ORDER BY table_name")).scalars().all()
print("init_db() 후 표 목록:")
print("  " + " · ".join(tables))

print()
print("seed_all()      →", seed_all())
print("seed_all() 다시 →", seed_all())      # 멱등성 검사 

with session_scope() as s:
    print("count_rows()    →", count_rows(s))